# Hyperparameter Tuning

## Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error


from sklearn.tree import DecisionTreeRegressor, plot_tree

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV


## Load Data

In [ ]:
# 1. Carga del dataset
df = pd.read_csv('data/house_pricing.csv')

In [ ]:
df.shape

## Train / Test

In [ ]:
# 1. Nos quedamos solo con las filas 'labeled' que contienen el SalePrice real
df_labeled = df[df['Split'] == 'labeled'].copy()
df_labeled = df_labeled.dropna(subset=['SalePrice'])

# 2. Separamos la variable objetivo
y = df_labeled['SalePrice']

# 3. Limpiamos X eliminando la variable objetivo y las columnas que no aportan valor predictivo
X = df_labeled.drop(columns=['SalePrice', 'Id', 'Split', 'Electtrical'], errors='ignore')

# 4. Convertimos los strings a 'category' de forma nativa para evitar el error en el HistGradient
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')

# 5. Partición Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

## Feature Engineering

Defineix els passos de preprocessament per als models que ho necessiten.

In [ ]:
# Identificación automática de tipos de datos para la Pipeline (para LR y RF)
categorical_features = X_train.select_dtypes(include=['category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Transformadores con imputación de nulos integrada para evitar fallos en modelos numéricos
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

## Cross-validation

In [ ]:
# Definición de la estrategia común de validación cruzada (10 Folds)
kf = KFold(n_splits=10, shuffle=True, random_state=42)

Dummy Regressor (Baseline de Referencia)

In [ ]:
dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dummy', DummyRegressor(strategy='mean'))
])

dummy_cv = cross_validate(dummy_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -dummy_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -dummy_cv['test_score'].mean().round(2))

LR: Linear Regresion

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LinearRegression())
])

lr_cv = cross_validate(lr_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -lr_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -lr_cv['test_score'].mean().round(2))

## Decision Tree

If this was not a pipeline, there would be no need to set `'dt__'` before hyperparameter names.

In [ ]:
# Random Forest (Normal)

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=42))
])

rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -rf_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -rf_cv['test_score'].mean().round(2))

In [ ]:
#Gráfica

# 1. Extraer los scores de cada fold y convertirlos a MAE positivo
train_mae_scores = -rf_cv['train_score']
val_mae_scores = -rf_cv['test_score']
folds = range(1, len(train_mae_scores) + 1)

# 2. Configurar el tamaño de la gráfica
plt.figure(figsize=(10, 6))

# 3. Graficar las líneas de Train y Validation fold por fold
plt.plot(folds, train_mae_scores, color='#1f77b4', marker='o', linestyle='-', linewidth=2, label=f'Train MAE (Media: {train_mae_scores.mean().round(2)})')
plt.plot(folds, val_mae_scores, color='#ff7f0e', marker='s', linestyle='--', linewidth=2, label=f'Validation MAE (Media: {val_mae_scores.mean().round(2)})')

# 4. Personalizar ejes, título y cuadrícula
plt.title('Rendimiento del Random Forest (Normal) en cada Fold de CV', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Fold', fontsize=12)
plt.ylabel('MAE (Error Absoluto Medio)', fontsize=12)
plt.xticks(folds)  # Asegura que se muestren los números del 1 al 10

# Añadir una línea horizontal con la media de validación como referencia visual
plt.axhline(y=val_mae_scores.mean(), color='red', linestyle=':', alpha=0.7, label='Media Validación')

# Estilo de la leyenda y diseño limpio
plt.legend(fontsize=11, loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

# 5. Mostrar la gráfica en tu notebook
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# 1. Entrenamos el pipeline una vez fuera de CV para poder extraer los árboles reales
rf_pipeline.fit(X_train, y_train)

# Extraemos el modelo Random Forest y el preprocesador del Pipeline
rf_model = rf_pipeline.named_steps['rf']
preprocessor_fitted = rf_pipeline.named_steps['preprocessor']

# Extraemos los nombres de las columnas transformadas (para saber qué variable divide cada nudo)
# Combinamos los nombres de variables numéricas y las categóricas tras el OneHotEncoder
encoded_cat_features = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features).tolist()
all_features_names = numeric_features + encoded_cat_features

# =========================================================================
# PARTE 1: Graficar los primeros niveles de un árbol individual del bosque
# =========================================================================
plt.figure(figsize=(20, 10))

# Tomamos el primer árbol del bosque (estimador 0)
# max_depth=3 limita la visualización a los primeros niveles para que sea legible
plot_tree(
    rf_model.estimators_[0],
    max_depth=3,
    feature_names=all_features_names,
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Estructura de Decisiones del Primer Árbol del Random Forest (Primeros 3 niveles)", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# =========================================================================
# PARTE 2: Calcular el número real de nudos y hojas de todo el bosque
# =========================================================================
# Extraemos el número de nudos de división y hojas finales de cada uno de los árboles
n_nodes = [estimator.tree_.node_count for estimator in rf_model.estimators_]
n_leaves = [estimator.tree_.n_leaves for estimator in rf_model.estimators_]

print("--- ANÁLISIS ESTRUCTURAL DEL BOSQUE ---")
print(f"Número de árboles en el bosque: {len(rf_model.estimators_)}")
print(f"Promedio de nudos totales por árbol: {int(np.mean(n_nodes))}")
print(f"Promedio de hojas (predicciones finales) por árbol: {int(np.mean(n_leaves))}")
print(f"Árbol más grande: {max(n_nodes)} nudos | Árbol más pequeño: {min(n_nodes)} nudos")

In [ ]:
# HistGradient Boosting (Normal)

# Funciona directo sobre el dataframe sin pasar por el preprocesor (OneHotEncoder)
gb = HistGradientBoostingRegressor(categorical_features='from_dtype', random_state=42)

gb_cv = cross_validate(gb, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -gb_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -gb_cv['test_score'].mean().round(2))

### Randomized Search

With a Decistion Tree.

As we're performing a randomized search, we can try wider ranges of hyperparameter values.

#### Round 1

In [ ]:
# Random Forest (Tuned)

rf_param_dist = {
    'rf__n_estimators': [50, 100, 200, 300],
    'rf__max_depth': [None, 10, 20, 30, 40],
    'rf__min_samples_split': range(2, 11),
    'rf__min_samples_leaf': range(1, 11),
    'rf__max_features': ['sqrt', 'log2', None]
}

rf_rs = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    random_state=42,
    n_jobs=-1
)
rf_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters:', rf_rs.best_params_)
print('CV Train MAE:', -rf_rs.cv_results_['mean_train_score'][rf_rs.best_index_].round(2))
print('CV Validation MAE:', -rf_rs.cv_results_['mean_test_score'][rf_rs.best_index_].round(2))

#### Round 2

In [ ]:
# HistGradient Boosting (Tuned)

gb_param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_iter': [50, 100, 150, 200],
    'max_depth': range(5, 51),
    'min_samples_leaf': range(10, 31),
    'l2_regularization': [0.0, 0.1, 1.0, 10.0]
}

gb_rs = RandomizedSearchCV(
    estimator=gb,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    random_state=42,
    n_jobs=-1
)
gb_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters:', gb_rs.best_params_)
print('CV Train MAE:', -gb_rs.cv_results_['mean_train_score'][gb_rs.best_index_].round(2))
print('CV Validation MAE:', -gb_rs.cv_results_['mean_test_score'][gb_rs.best_index_].round(2))

**Comparación Final de Modelos**

In [ ]:
resultados = pd.DataFrame({
    'Model': [
        'Dummy',
        'Linear Regression',
        'Random Forest (Base)',
        'HistGradient Boosting (Base)',
        'Random Forest (Tuned)',
        'HistGradient Boosting (Tuned)'
    ],
    'Validation MAE': [
        -dummy_cv['test_score'].mean().round(2),
        -lr_cv['test_score'].mean().round(2),
        -rf_cv['test_score'].mean().round(2),
        -gb_cv['test_score'].mean().round(2),
        -rf_rs.cv_results_['mean_test_score'][rf_rs.best_index_].round(2),
        -gb_rs.cv_results_['mean_test_score'][gb_rs.best_index_].round(2)
    ]
})

print(resultados.to_string(index=False))

Creando el fichero con LeaderBorad

In [ ]:
# 1. Filtramos las filas del leaderboard conservando el 'Id' original
df_leaderboard = df[df['Split'] == 'leaderboard'].copy()

# 2. Creamos la matriz de características X_leaderboard
X_leaderboard = df_leaderboard.drop(columns=['SalePrice', 'Id', 'Split', 'Electtrical'], errors='ignore')

# 3. Conversión a 'category' para evitar errores de tipo string
for col in X_leaderboard.select_dtypes(include=['object']).columns:
    X_leaderboard[col] = X_leaderboard[col].astype('category')

# 4. El modelo ganador realiza las predicciones
predicciones_leaderboard = gb_rs.predict(X_leaderboard)

# 5. Estructuramos el DataFrame final con el nombre exigido: 'prediction'
submission_df = pd.DataFrame({
    'Id': df_leaderboard['Id'],
    'prediction': predicciones_leaderboard.round(2)  # <--- Cambiado de 'SalePrice' a 'prediction'
})

# 6. Exportamos el nuevo archivo CSV corregido
submission_df.to_csv('house_pricing_predictions_levi_1.csv', index=False)

print("--- ARCHIVO GENERADO CON ÉXITO ---")
print(submission_df.head())